## Patient Journey - PX
#### Data source used
* PHCDW.PHCDW_PUB_CAR.PUB_CAR_IQV_LAAD_PX_FCT (PHCDW.PHCDW_DSAA.IQV_LAAD_PX_VW was used before 27th Oct, 2022)
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE

#### Steps:
1. Query px claims data
2. Get first dialysis date for each patient
3. Save table to databricks

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../../00_cohort_config"

In [0]:


query = """
      SELECT DISTINCT PATIENT_ID,
                      min(SERVICE_DATE) as FIRST_DIALYSIS_DATE,    
                      1 as DIALYSIS_FLAG
                      --PX.PROCEDURE_CODE,
                      --PROCEDURE_DESCRIPTION
                      
      FROM (select PATIENT_ID, SERVICE_DATE, PROCEDURE_CODE from PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX)  PX
      JOIN 
            (   SELECT DISTINCT  PROCEDURE_CODE,PROCEDURE_DESCRIPTION
                FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE 
                WHERE UPPER(PROCEDURE_DESCRIPTION) LIKE '%DIALYSIS%'
            ) PXCD
      ON PX.PROCEDURE_CODE = PXCD.PROCEDURE_CODE
      group by PATIENT_ID, DIALYSIS_FLAG
   """  
df_px = Get_Data_Snowflakes(query)
df_px.display()

In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_px.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_px_validation")